# SalesInsight PY

*Análise e visualização de dados de vendas*

**Autor:** Bruno Miguel Corrêa

**Curso:** Desenvolvimento de IA para Análise Preditiva — SENAI/SC


## Visão Geral

O SalesInsight PY analisa vendas realizadas entre janeiro e junho de 2025 para identificar a evolução mensal da receita, os produtos, categorias e regiões de maior desempenho e os clientes mais valiosos.

A análise abrange inspeção, limpeza, transformação e agregação dos dados com Pandas e NumPy. Os resultados são apresentados em visualizações produzidas com Matplotlib e Seaborn e consolidados em um relatório JSON.

## 1. Configuração do Ambiente

As versões das principais tecnologias são registradas para garantir a rastreabilidade e facilitar a reprodução da análise.

In [1]:
import platform
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sales_insight.dicionario_dados import DICIONARIO_DADOS
from sales_insight.apresentacao import exibir_tabela

print("=== Ambiente de execução ===")
print(f"Python: {platform.python_version()}")
print(f"Pandas: {pd.__version__}")
print(f"NumPy:  {np.__version__}")

=== Ambiente de execução ===
Python: 3.13.14
Pandas: 3.0.5
NumPy:  2.5.1


## 2. Carregamento e Inspeção dos Dados

### `RF01` — Carregamento do Dataset

O `vendas.csv` foi construído a partir do [E-commerce Analytics Dataset Brazil](https://www.kaggle.com/datasets/joocarlosjr/e-commerce-analytics-dataset-brazil). A base foi consolidada, adaptada e enriquecida com informações comerciais e logísticas.

Foram mantidas inconsistências controladas para permitir a aplicação das técnicas de limpeza e validação. O arquivo bruto permanece preservado em `data/raw`.

In [2]:
caminho_dados = Path("../data/raw/vendas.csv")

df_bruto = pd.read_csv(caminho_dados)
df = df_bruto.copy()

### `RF02` — Inspeção Estrutural

Antes do tratamento, a estrutura e a qualidade inicial da base são verificadas por meio de uma amostra dos registros, do dicionário de dados, das dimensões, dos tipos das colunas e dos valores ausentes.

In [3]:
display(df.head(4).style.hide(axis="index"))

id_venda,data_venda,id_cliente,nome_cliente,cidade,estado,regiao,id_produto,produto,categoria,quantidade,preco_unitario,desconto,previsao_entrega,data_entrega
ORD00001,2025-01-01,Cliente_008,Thales Pereira,Cascavel,PR,Sul,P0005,"Fone de Ouvido Sem Fio TWS, PHILIPS",Áudio,1.000000,124.910000,0.032700,2025-01-07,2025-01-09
ORD00002,2025-01-01,Cliente_011,Julia Ribeiro,Santa Maria,RS,Sul,P0006,Fone de ouvido Sem Fio QCY T27,Áudio,4.000000,129.860000,0.047700,2025-01-04,2025-01-08
ORD00003,2025-01-01,Cliente_019,Gabriela da Paz,Uberlândia,MG,Sudeste,P0002,Samsung Galaxy Tab S6 Lite,Smartphones e Tablets,2.000000,1799.010000,0.148100,2025-01-05,2025-01-04
ORD00004,2025-01-01,Cliente_019,Gabriela da Paz,Uberlândia,MG,Sudeste,P0004,Carregador Turbo Tipo-C 50w,Acessórios Mobile,4.000000,53.900000,0.140200,2025-01-04,2025-01-04


#### Dicionário dos Dados


In [4]:
exibir_tabela(pd.DataFrame(DICIONARIO_DADOS))

Coluna,Descrição
id_venda,Identificador único da venda.
data_venda,Data em que a venda foi realizada.
id_cliente,Identificador do cliente associado à venda.
nome_cliente,Nome do cliente.
cidade,Cidade de residência do cliente.
estado,Estado de residência do cliente.
regiao,Região geográfica associada ao cliente.
id_produto,Identificador do produto vendido.
produto,Nome do produto vendido.
categoria,Categoria à qual o produto pertence.


#### Dimensões, Tipos e Valores Ausentes


In [5]:
def resumo_estrutural(dataframe):
    """
    Resume os tipos e os valores ausentes de cada coluna.
    """
    return pd.DataFrame(
        {
            "coluna": dataframe.columns,
            "tipo": dataframe.dtypes.astype(str).values,
            "valores_ausentes": dataframe.isna().sum().values,
            "percentual_ausente": dataframe.isna().mean().values * 100,
        }
    )


linhas, colunas = df.shape
resumo_inicial = resumo_estrutural(df)

print(f"Dimensões: {linhas} linhas × {colunas} colunas")

exibir_tabela(resumo_inicial, formatos={"percentual_ausente": "{:.2f}%"})

Dimensões: 5732 linhas × 15 colunas


coluna,tipo,valores_ausentes,percentual_ausente
id_venda,str,0,0.00%
data_venda,str,63,1.10%
id_cliente,str,0,0.00%
nome_cliente,str,0,0.00%
cidade,str,0,0.00%
estado,str,0,0.00%
regiao,str,0,0.00%
id_produto,str,0,0.00%
produto,str,0,0.00%
categoria,str,0,0.00%


#### Diagnóstico Inicial

A base bruta contém 5.732 registros e 15 colunas. As colunas `data_venda`, `previsao_entrega` e `data_entrega` foram carregadas como texto e precisam ser convertidas para `datetime`.

Foram identificados valores ausentes em `data_venda` (63), `quantidade` (229) e `preco_unitario` (86). Esses pontos definem o escopo da limpeza realizada no RF03.


## 3. Limpeza e Tratamento dos Dados

### `RF03` — Padronização e Validação

Antes das análises, os identificadores, textos, datas e valores numéricos são verificados e padronizados. Registros sem informações essenciais são removidos para evitar que dados incompletos comprometam os cálculos posteriores.

In [6]:
registros_iniciais = len(df)

padroes_identificadores = {
    "id_venda": re.compile(r"^ORD\d{5}$"),
    "id_cliente": re.compile(r"^Cliente_\d{3}$"),
    "id_produto": re.compile(r"^P\d{4}$"),
}


def validar_identificadores(dataframe, padroes):
    """
    Retorna o resultado da validação dos identificadores.
    """
    resultados = []

    for coluna, padrao in padroes.items():
        validos = dataframe[coluna].astype("string").str.fullmatch(padrao, na=False)

        resultados.append(
            {
                "coluna": coluna,
                "valores_ausentes": int(dataframe[coluna].isna().sum()),
                "fora_do_padrao": int((~validos).sum()),
                "duplicados": (
                    int(dataframe[coluna].duplicated().sum())
                    if coluna == "id_venda"
                    else pd.NA
                ),
            }
        )

    return pd.DataFrame(resultados)


validacao_ids_antes = validar_identificadores(
    df,
    padroes_identificadores,
)

ids_clientes_corrigidos = int(
    validacao_ids_antes.loc[
        validacao_ids_antes["coluna"] == "id_cliente",
        "fora_do_padrao",
    ].iloc[0]
)

exibir_tabela(validacao_ids_antes)

coluna,valores_ausentes,fora_do_padrao,duplicados
id_venda,0,0,0
id_cliente,0,226,
id_produto,0,0,


#### 3.1 Padronização dos Identificadores e Textos

A validação inicial encontrou 226 identificadores de clientes fora do formato `Cliente_000`. Esses valores são normalizados com expressão regular, enquanto as demais colunas textuais recebem a remoção de espaços excedentes.

In [7]:
def normalizar_id_cliente(valor):
    """Padroniza o identificador para o formato Cliente_000."""
    return re.sub(
        r"^cliente\D*(\d{3})\D*$",
        r"Cliente_\1",
        str(valor).strip(),
        flags=re.IGNORECASE,
    )


df["id_cliente"] = df["id_cliente"].apply(normalizar_id_cliente)

colunas_textuais = [
    "nome_cliente",
    "cidade",
    "estado",
    "regiao",
    "produto",
    "categoria",
]

resultado_padronizacao = []

for coluna in colunas_textuais:
    original = df[coluna].copy()
    padronizado = original.str.strip()

    resultado_padronizacao.append(
        {
            "coluna": coluna,
            "unicos_antes": original.nunique(),
            "unicos_depois": padronizado.nunique(),
            "espacos_corrigidos": int(original.ne(padronizado).sum()),
        }
    )

    df[coluna] = padronizado

resumo_padronizacao = pd.DataFrame(resultado_padronizacao)

validacao_ids_depois = validar_identificadores(
    df,
    padroes_identificadores,
)

exibir_tabela(validacao_ids_depois)
print()
exibir_tabela(resumo_padronizacao)

coluna,valores_ausentes,fora_do_padrao,duplicados
id_venda,0,0,0
id_cliente,0,0,
id_produto,0,0,


coluna,unicos_antes,unicos_depois,espacos_corrigidos
nome_cliente,530,530,0
cidade,105,105,0
estado,22,22,0
regiao,4,4,0
produto,72,27,103
categoria,15,5,69


#### 3.2 Conversão das Datas

As três colunas temporais são convertidas para `datetime`. Valores incompatíveis são transformados em `NaT` e contabilizados, permitindo verificar se houve perda de informações durante a conversão.

In [8]:
colunas_datas = [
    "data_venda",
    "previsao_entrega",
    "data_entrega",
]

datas_originais = df[colunas_datas].copy()

df[colunas_datas] = df[colunas_datas].apply(
    pd.to_datetime,
    errors="coerce",
)

falhas_conversao = (df[colunas_datas].isna() & datas_originais.notna()).sum()

validacao_datas = pd.DataFrame(
    {
        "coluna": colunas_datas,
        "tipo_final": [str(df[coluna].dtype) for coluna in colunas_datas],
        "falhas_conversao": falhas_conversao.values,
    }
)

exibir_tabela(validacao_datas)

coluna,tipo_final,falhas_conversao
data_venda,datetime64[us],0
previsao_entrega,datetime64[us],0
data_entrega,datetime64[us],0


#### 3.3 Tratamento das Ausências e Validação Numérica

Os registros com ausência em `data_venda`, `quantidade` ou `preco_unitario` serão removidos. Como existem linhas com ausência em mais de uma dessas colunas, a remoção considera os registros únicos afetados.


In [9]:
colunas_criticas = [
    "data_venda",
    "quantidade",
    "preco_unitario",
]

ausencias_criticas = (
    df[colunas_criticas]
    .isna()
    .sum()
    .rename_axis("coluna")
    .reset_index(name="valores_ausentes")
)

mascara_remocao = df[colunas_criticas].isna().any(axis=1)

total_removidos = int(mascara_remocao.sum())

df = df.loc[~mascara_remocao].copy()

exibir_tabela(ausencias_criticas)

print(f"Registros únicos removidos: {total_removidos}")

coluna,valores_ausentes
data_venda,63
quantidade,229
preco_unitario,86


Registros únicos removidos: 371


In [10]:
validacoes_numericas = {
    "quantidade_inteira": bool((df["quantidade"] % 1 == 0).all()),
    "quantidade_positiva": bool((df["quantidade"] > 0).all()),
    "preco_positivo": bool((df["preco_unitario"] > 0).all()),
    "desconto_valido": bool(df["desconto"].between(0, 1).all()),
}

if not all(validacoes_numericas.values()):
    raise ValueError("Foram encontrados valores numéricos inválidos.")

df["quantidade"] = df["quantidade"].astype("int64")

colunas_numericas = [
    "quantidade",
    "preco_unitario",
    "desconto",
]

validacao_numerica = pd.DataFrame(
    {
        "coluna": colunas_numericas,
        "tipo": (df[colunas_numericas].dtypes.astype(str).values),
        "valor_minimo": [df[coluna].min() for coluna in colunas_numericas],
        "valor_maximo": [df[coluna].max() for coluna in colunas_numericas],
        "regra_validada": [
            (
                validacoes_numericas["quantidade_inteira"]
                and validacoes_numericas["quantidade_positiva"]
            ),
            validacoes_numericas["preco_positivo"],
            validacoes_numericas["desconto_valido"],
        ],
    }
)

exibir_tabela(
    validacao_numerica,
)

coluna,tipo,valor_minimo,valor_maximo,regra_validada
quantidade,int64,1.000000,4.000000,True
preco_unitario,float64,17.900000,4604.000000,True
desconto,float64,0.000100,0.350000,True


#### Resultado da Limpeza


In [11]:
registros_finais = len(df)

espacos_corrigidos = {
    linha.coluna: linha.espacos_corrigidos for linha in resumo_padronizacao.itertuples()
}

relatorio_limpeza = {
    "registros_iniciais": registros_iniciais,
    "ids_cliente_normalizados": ids_clientes_corrigidos,
    "espacos_produto_corrigidos": espacos_corrigidos["produto"],
    "espacos_categoria_corrigidos": espacos_corrigidos["categoria"],
    "registros_removidos": total_removidos,
    "registros_finais": registros_finais,
}

tabela_relatorio_limpeza = pd.DataFrame(
    relatorio_limpeza.items(),
    columns=["indicador", "valor"],
)

exibir_tabela(tabela_relatorio_limpeza)

exibir_tabela(resumo_estrutural(df), formatos={"percentual_ausente": "{:.2f}%"})

indicador,valor
registros_iniciais,5732
ids_cliente_normalizados,226
espacos_produto_corrigidos,103
espacos_categoria_corrigidos,69
registros_removidos,371
registros_finais,5361


coluna,tipo,valores_ausentes,percentual_ausente
id_venda,str,0,0.00%
data_venda,datetime64[us],0,0.00%
id_cliente,str,0,0.00%
nome_cliente,str,0,0.00%
cidade,str,0,0.00%
estado,str,0,0.00%
regiao,str,0,0.00%
id_produto,str,0,0.00%
produto,str,0,0.00%
categoria,str,0,0.00%


Foram normalizados 226 identificadores de clientes, além de 103 ocorrências com espaços excedentes em produtos e 69 em categorias. As datas foram convertidas sem falhas de formato.

A remoção das ausências críticas eliminou 371 registros únicos, mantendo 5.361 vendas válidas. Após o tratamento, as quantidades são inteiras e positivas, os preços são positivos e os descontos permanecem no intervalo esperado.

## 4. Transformação e Criação de Variáveis

### `RF04` — Colunas Derivadas e Transformações Condicionais

Com a base limpa e validada, são criadas variáveis financeiras, temporais, logísticas e categóricas que servirão de apoio às análises seguintes.

#### 4.1 Variáveis Financeiras

O preço líquido unitário é obtido após a aplicação do desconto e arredondado para centavos. A receita total resulta da multiplicação desse valor pela quantidade vendida, enquanto `valor_desconto` registra a redução concedida na transação.

Com base na receita, cada venda é classificada como `Baixo Valor`, `Médio Valor` ou `Alto Valor` por meio de `np.select`.

In [12]:
preco_liquido_unitario = (df["preco_unitario"] * (1 - df["desconto"])).round(2)

df["receita_total"] = (df["quantidade"] * preco_liquido_unitario).round(2)

df["valor_desconto"] = (
    df["quantidade"] * df["preco_unitario"] - df["receita_total"]
).round(2)

condicoes_receita = [
    df["receita_total"] < 500,
    df["receita_total"].between(500, 4999.99),
    df["receita_total"] >= 5000,
]

faixas_receita = [
    "Baixo Valor",
    "Médio Valor",
    "Alto Valor",
]

df["faixa_receita_item"] = np.select(
    condicoes_receita,
    faixas_receita,
    default="Não Classificado",
)

exibir_tabela(
    df[
        [
            "quantidade",
            "preco_unitario",
            "desconto",
            "valor_desconto",
            "receita_total",
            "faixa_receita_item",
        ]
    ].head(5),
    formatos={
        "preco_unitario": "R$ {:,.2f}",
        "desconto": "{:.2%}",
        "valor_desconto": "R$ {:,.2f}",
        "receita_total": "R$ {:,.2f}",
    },
)

quantidade,preco_unitario,desconto,valor_desconto,receita_total,faixa_receita_item
1,R$ 124.91,3.27%,R$ 4.08,R$ 120.83,Baixo Valor
4,R$ 129.86,4.77%,R$ 24.76,R$ 494.68,Baixo Valor
2,"R$ 1,799.01",14.81%,R$ 532.86,"R$ 3,065.16",Médio Valor
4,R$ 53.90,14.02%,R$ 30.24,R$ 185.36,Baixo Valor
2,R$ 389.90,11.26%,R$ 87.80,R$ 692.00,Médio Valor


#### 4.2 Variáveis Temporais

A data da venda é decomposta em ano, trimestre, número e nome do mês. Essas variáveis permitem organizar cronologicamente os resultados e realizar agregações por período.

In [13]:
meses = {
    1: "Janeiro",
    2: "Fevereiro",
    3: "Março",
    4: "Abril",
    5: "Maio",
    6: "Junho",
    7: "Julho",
    8: "Agosto",
    9: "Setembro",
    10: "Outubro",
    11: "Novembro",
    12: "Dezembro",
}

df["mes"] = df["data_venda"].dt.month
df["mes_venda"] = df["mes"].map(meses)
df["trimestre"] = "Q" + df["data_venda"].dt.quarter.astype(str)
df["ano"] = df["data_venda"].dt.year

periodos_identificados = (
    df[
        [
            "ano",
            "trimestre",
            "mes",
            "mes_venda",
        ]
    ]
    .drop_duplicates()
    .sort_values(["ano", "mes"])
    .reset_index(drop=True)
)

print(
    "Período analisado: "
    f"{df['data_venda'].min():%d/%m/%Y} a "
    f"{df['data_venda'].max():%d/%m/%Y}"
)

exibir_tabela(periodos_identificados)

Período analisado: 01/01/2025 a 30/06/2025


ano,trimestre,mes,mes_venda
2025,Q1,1,Janeiro
2025,Q1,2,Fevereiro
2025,Q1,3,Março
2025,Q2,4,Abril
2025,Q2,5,Maio
2025,Q2,6,Junho


#### 4.3 Variáveis Logísticas

A diferença entre a entrega efetiva e a previsão determina o desvio em dias. Valores negativos indicam antecipação, zero representa entrega no prazo e valores positivos identificam atraso.

In [14]:
df["desvio_entrega_dias"] = (df["data_entrega"] - df["previsao_entrega"]).dt.days

df["atrasado"] = df["desvio_entrega_dias"] > 0

resumo_entregas = pd.DataFrame(
    {
        "situacao": [
            "Antecipada",
            "No prazo",
            "Atrasada",
        ],
        "numero_entregas": [
            int((df["desvio_entrega_dias"] < 0).sum()),
            int((df["desvio_entrega_dias"] == 0).sum()),
            int(df["atrasado"].sum()),
        ],
    }
)

exibir_tabela(resumo_entregas)

situacao,numero_entregas
Antecipada,2364
No prazo,1693
Atrasada,1304


#### 4.4 Dataset Processado

Após as transformações, as colunas são organizadas por domínio e a base resultante é exportada para `data/processed/vendas_processado.csv`. Esse arquivo será utilizado pelo notebook de visualizações.

In [15]:
ordem_colunas = [
    "id_venda",
    "data_venda",
    "ano",
    "trimestre",
    "mes",
    "mes_venda",
    "id_cliente",
    "nome_cliente",
    "cidade",
    "estado",
    "regiao",
    "id_produto",
    "produto",
    "categoria",
    "quantidade",
    "preco_unitario",
    "desconto",
    "valor_desconto",
    "receita_total",
    "faixa_receita_item",
    "previsao_entrega",
    "data_entrega",
    "desvio_entrega_dias",
    "atrasado",
]

df = df[ordem_colunas]

pasta_processados = Path("../data/processed")
pasta_processados.mkdir(
    parents=True,
    exist_ok=True,
)

caminho_saida = pasta_processados / "vendas_processado.csv"

df.to_csv(
    caminho_saida,
    index=False,
    encoding="utf-8-sig",
)

print(f"Dataset processado: " f"{df.shape[0]} registros × " f"{df.shape[1]} colunas")

print(f"Arquivo salvo em: {caminho_saida}")

Dataset processado: 5361 registros × 24 colunas
Arquivo salvo em: ../data/processed/vendas_processado.csv


Foram criadas nove variáveis derivadas: três financeiras, quatro temporais e duas logísticas. O dataset processado contém 5.361 registros e 24 colunas, prontos para as análises e visualizações.

## 5. Métricas Agregadas de Vendas

### `RF05` — Agregações com `groupby`

As vendas são agrupadas por mês, produto, categoria e região para comparar o desempenho comercial sob diferentes perspectivas. A função `calcular_metricas()` reúne os quatro conjuntos de resultados em um dicionário de DataFrames.

In [16]:
def calcular_metricas(dataframe):
    """Calcula as métricas agregadas e retorna um dicionário de DataFrames."""
    por_mes = (
        dataframe.groupby(
            ["mes", "mes_venda"],
            as_index=False,
        )
        .agg(
            receita_total=("receita_total", "sum"),
            unidades_vendidas=("quantidade", "sum"),
            numero_vendas=("id_venda", "nunique"),
        )
        .sort_values("mes")
        .reset_index(drop=True)
    )

    por_produto = (
        dataframe.groupby("produto", as_index=False)
        .agg(
            receita_total=("receita_total", "sum"),
        )
        .sort_values(
            "receita_total",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    por_categoria = (
        dataframe.groupby("categoria", as_index=False)
        .agg(
            receita_total=("receita_total", "sum"),
        )
        .sort_values(
            "receita_total",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    por_regiao = (
        dataframe.groupby("regiao", as_index=False)
        .agg(
            receita_total=("receita_total", "sum"),
            numero_vendas=("id_venda", "nunique"),
            ticket_medio=("receita_total", "mean"),
        )
        .sort_values(
            "receita_total",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    por_regiao["ticket_medio"] = por_regiao["ticket_medio"].round(2)

    return {
        "por_mes": por_mes,
        "por_produto": por_produto,
        "por_categoria": por_categoria,
        "por_regiao": por_regiao,
    }


metricas_gerais = calcular_metricas(df)

#### 5.1 Desempenho Mensal

Como a receita, a quantidade vendida e o número de vendas se distribuíram entre os meses?


In [17]:
exibir_tabela(
    metricas_gerais["por_mes"],
    formatos={
        "receita_total": "R$ {:,.2f}",
    },
)

mes,mes_venda,receita_total,unidades_vendidas,numero_vendas
1,Janeiro,"R$ 1,299,266.31",1968,824
2,Fevereiro,"R$ 1,591,691.83",2063,855
3,Março,"R$ 2,291,715.46",2443,991
4,Abril,"R$ 2,004,674.59",2241,900
5,Maio,"R$ 1,613,016.68",2081,880
6,Junho,"R$ 1,537,652.30",2131,911


#### 5.2 Produtos com Maior Receita

Quais produtos apresentaram as cinco maiores receitas acumuladas?


In [18]:
top_5_produtos = metricas_gerais["por_produto"].head(5)

exibir_tabela(
    top_5_produtos,
    formatos={
        "receita_total": "R$ {:,.2f}",
    },
)

produto,receita_total
ACER Notebook Gamer Nitro,"R$ 1,848,972.94"
"Projetor Smart Epson EpiqVision, FULL HD","R$ 1,244,636.36"
Samsung Galaxy A36,"R$ 850,244.53"
Samsung Galaxy Tab S6 Lite,"R$ 839,934.77"
Projetor EPSON Powerlite Wide Screen,"R$ 801,184.04"


#### 5.3 Receita por Categoria

Como a receita total se distribuiu entre as categorias de produtos?


In [19]:
exibir_tabela(
    metricas_gerais["por_categoria"],
    formatos={
        "receita_total": "R$ {:,.2f}",
    },
)

categoria,receita_total
Informática,"R$ 3,049,354.74"
TV e Projeção,"R$ 3,022,977.33"
Áudio,"R$ 2,140,350.33"
Smartphones e Tablets,"R$ 2,017,909.85"
Acessórios Mobile,"R$ 107,424.92"


#### 5.4 Desempenho Regional

Como as regiões se comparam em receita total, número de vendas e ticket médio?


In [20]:
exibir_tabela(
    metricas_gerais["por_regiao"],
    formatos={
        "receita_total": "R$ {:,.2f}",
        "ticket_medio": "R$ {:,.2f}",
    },
)

regiao,receita_total,numero_vendas,ticket_medio
Sudeste,"R$ 2,829,006.77",1435,"R$ 1,971.43"
Sul,"R$ 2,776,130.94",1477,"R$ 1,879.57"
Norte,"R$ 2,393,107.34",1232,"R$ 1,942.46"
Nordeste,"R$ 2,339,772.12",1217,"R$ 1,922.57"


Março liderou os três indicadores mensais, com receita de R$ 2,29 milhões, 2.443 unidades vendidas e 991 vendas.

O ACER Notebook Gamer Nitro apresentou a maior receita entre os produtos, com R$ 1,85 milhão. Entre as categorias, Informática ocupou a primeira posição, seguida de TV e Projeção.

O Sudeste registrou a maior receita regional e o maior ticket médio. O Sul teve o maior número de vendas, mas o menor ticket médio, mostrando que um volume maior de transações não implica necessariamente maior receita por venda.

## 6. Segmentação de Clientes

### `RF06` — Classificação por Nível de Gasto

As vendas serão agrupadas por cliente e classificadas de acordo com a receita acumulada.

| Gasto acumulado               | Segmento |
| ----------------------------- | -------- |
| Abaixo de $\text{R\$}$ 5.000,00         | Bronze   |
| De $\text{R\$}$ 5.000,00 a $\text{R\$}$ 15.000,00 | Prata    |
| Acima de $\text{R\$}$ 15.000,00         | Ouro     |


In [21]:
def segmentar_clientes(dataframe):
    """Agrupa e classifica os clientes pelo gasto acumulado."""
    clientes = dataframe.groupby(
        ["id_cliente", "nome_cliente"],
        as_index=False,
    ).agg(
        gasto_total=("receita_total", "sum"),
    )

    clientes["gasto_total"] = clientes["gasto_total"].round(2)

    clientes["segmento"] = clientes["gasto_total"].apply(
        lambda gasto: (
            "Bronze" if gasto < 5_000 else "Prata" if gasto <= 15_000 else "Ouro"
        )
    )

    return clientes.sort_values(
        "gasto_total",
        ascending=False,
    ).reset_index(drop=True)


clientes_segmentados = segmentar_clientes(df)

#### 6.1 Distribuição por Segmento

Como os clientes estão distribuídos entre os segmentos Ouro, Prata e Bronze?

In [22]:
ordem_segmentos = [
    "Ouro",
    "Prata",
    "Bronze",
]

distribuicao_segmentos = (
    clientes_segmentados["segmento"]
    .value_counts()
    .reindex(
        ordem_segmentos,
        fill_value=0,
    )
    .rename_axis("segmento")
    .reset_index(name="numero_clientes")
)

distribuicao_segmentos["percentual"] = (
    distribuicao_segmentos["numero_clientes"] / len(clientes_segmentados) * 100
).round(2)

exibir_tabela(
    distribuicao_segmentos,
    formatos={
        "percentual": "{:.2f}%",
    },
)

segmento,numero_clientes,percentual
Ouro,245,45.88%
Prata,182,34.08%
Bronze,107,20.04%


#### 6.2 Clientes com Maior Gasto

Quais são os dez clientes com maior gasto acumulado no período?

In [23]:
top_10_clientes = clientes_segmentados.head(10)

exibir_tabela(
    top_10_clientes,
    formatos={
        "gasto_total": "R$ {:,.2f}",
    },
)

id_cliente,nome_cliente,gasto_total,segmento
Cliente_371,Ana Luiza Nunes,"R$ 116,525.85",Ouro
Cliente_165,Maria Fernanda Teixeira,"R$ 111,976.65",Ouro
Cliente_394,Eduardo Oliveira,"R$ 108,436.37",Ouro
Cliente_008,Thales Pereira,"R$ 96,223.86",Ouro
Cliente_360,Mariane Campos,"R$ 92,852.62",Ouro
Cliente_300,Dra. Maria Sophia Nogueira,"R$ 89,455.87",Ouro
Cliente_218,Isabel Pereira,"R$ 89,303.07",Ouro
Cliente_337,Benício Fernandes,"R$ 87,821.42",Ouro
Cliente_266,Caroline Carvalho,"R$ 87,502.24",Ouro
Cliente_357,Amanda Araújo,"R$ 81,303.37",Ouro


Foram classificados 534 clientes. O segmento Ouro reúne 245 clientes (45,88%), seguido por Prata, com 182 (34,08%), e Bronze, com 107 (20,04%).

Ana Luiza Nunes apresentou o maior gasto acumulado, com R$ 116.525,85. Todos os clientes do Top 10 pertencem ao segmento Ouro.

## 7. Operações Numéricas com NumPy

### `RF07` — Vetorização, Broadcasting e Filtragem

A receita será convertida de `Series` para um array NumPy. Sobre esse array serão aplicadas funções de agregação, operações vetorizadas com escalares e filtragem por máscara booleana.


In [24]:
receitas_array = df["receita_total"].to_numpy()


def calcular_estatisticas_numpy(valores):
    """Calcula estatísticas agregadas sobre um array NumPy."""
    return {
        "media": float(np.mean(valores)),
        "mediana": float(np.median(valores)),
        "desvio_padrao": float(np.std(valores, ddof=0)),
        "soma": float(np.sum(valores)),
        "minimo": float(np.min(valores)),
        "maximo": float(np.max(valores)),
    }


estatisticas_receita = calcular_estatisticas_numpy(receitas_array)

resumo_array = pd.DataFrame(
    [
        {
            "tipo": str(receitas_array.dtype),
            "dimensoes": receitas_array.ndim,
            "elementos": receitas_array.size,
        }
    ]
)

tabela_estatisticas = pd.DataFrame(
    estatisticas_receita.items(),
    columns=["medida", "valor"],
)

exibir_tabela(resumo_array)

exibir_tabela(
    tabela_estatisticas,
    formatos={
        "valor": "R$ {:,.2f}",
    },
)

tipo,dimensoes,elementos
float64,1,5361


medida,valor
media,"R$ 1,928.37"
mediana,R$ 979.70
desvio_padrao,"R$ 2,530.21"
soma,"R$ 10,338,017.17"
minimo,R$ 13.83
maximo,"R$ 18,316.56"


#### 7.1 Escalonamento Vetorizado

Para demonstrar broadcasting, as receitas são escalonadas para o intervalo entre 0 e 1 por meio de operações entre o array e valores escalares. O resultado é apenas demonstrativo e não altera os valores armazenados no DataFrame.

$$x_{\text{normalizado}} = \dfrac{x - x_{\text{min}}}{x_{\text{max}}-x_{\text{min}}}$$

In [25]:
valor_minimo = estatisticas_receita["minimo"]
valor_maximo = estatisticas_receita["maximo"]

if valor_maximo == valor_minimo:
    raise ValueError("Não é possível escalonar um array sem variação.")

receitas_escalonadas = (receitas_array - valor_minimo) / (valor_maximo - valor_minimo)

print(
    "Intervalo após o escalonamento: "
    f"{receitas_escalonadas.min():.1f} a "
    f"{receitas_escalonadas.max():.1f}"
)

Intervalo após o escalonamento: 0.0 a 1.0


#### 7.2 Filtragem Acima da Média

Uma comparação vetorizada gera a máscara booleana utilizada para selecionar e contabilizar as vendas com receita superior à média.

In [26]:
media_receita = estatisticas_receita["media"]

mascara_acima_media = receitas_array > media_receita

receitas_acima_media = receitas_array[mascara_acima_media]

quantidade_vendas = receitas_array.size
quantidade_acima_media = receitas_acima_media.size

percentual_acima_media = quantidade_acima_media / quantidade_vendas * 100

resumo_filtro = pd.DataFrame(
    [
        {
            "total_vendas": quantidade_vendas,
            "vendas_acima_media": quantidade_acima_media,
            "percentual": percentual_acima_media,
        }
    ]
)

exibir_tabela(
    resumo_filtro,
    formatos={
        "percentual": "{:.2f}%",
    },
)

total_vendas,vendas_acima_media,percentual
5361,1720,32.08%


O array contém 5.361 receitas do tipo `float64`. A receita média foi de $\text{R\$}$ 1.928,37 e a mediana de $\text{R\$}$ 979,70. O desvio-padrão populacional, calculado por `np.std()` com `ddof=0`, foi de $\text{R\$}$ 2.530,21.

O escalonamento confirmou o intervalo entre 0 e 1. A máscara booleana identificou 1.720 vendas acima da média, equivalentes a 32,08% das transações.


## 8. Visualização dos Resultados

### `RF08` — Matplotlib e Seaborn

As figuras foram produzidas no notebook de visualizações e exportadas para `reports/figures`. Esta seção reúne os resultados gráficos e suas interpretações.


#### 8.1 Evolução da Receita Mensal

Como a receita se comportou ao longo do período analisado?

![Receita total por mês](../reports/figures/receita_por_mes.png)

A receita aumentou de $\text{R\$}$ 1,30 milhão em janeiro para $\text{R\$}$ 2,29 milhões em março, maior resultado do período. Após o pico, recuou continuamente até atingir $\text{R\$}$ 1,54 milhão em junho.


#### 8.2 Produtos com Maior Receita

Quais produtos apresentaram as maiores receitas acumuladas?

![Produtos com maior receita](../reports/figures/top_produtos.png)

O ACER Notebook Gamer Nitro liderou o ranking, com $\text{R\$}$ 1,85 milhão, seguido pelo Projetor Smart Epson EpiqVision, com $\text{R\$}$ 1,24 milhão. Os demais produtos do Top 5 permaneceram abaixo de $\text{R\$}$ 860 mil.


#### 8.3 Distribuição das Receitas por Transação

Como os valores das vendas estão distribuídos?

![Distribuição das receitas por transação](../reports/figures/distribuicao_receitas.png)

A maior parte das vendas está concentrada nas faixas de menor receita, enquanto uma quantidade menor de transações alcança valores elevados, formando uma cauda à direita.

Essa distribuição eleva a média para $\text{R\$}$ 1.928,37, quase o dobro da mediana de $\text{R\$}$ 979,70. O desvio-padrão de $\text{R\$}$ 2.530,21, superior à própria média, evidencia a ampla variação entre os valores das transações.

#### 8.4 Desempenho Comercial das Cidades

Quais cidades concentram o maior número de vendas e as maiores receitas?

![Cidades com mais vendas e receita](../reports/figures/top_cidades_vendas_receita.png)

Ribeirão Preto e Cascavel ocupam, respectivamente, a primeira e a segunda posições nos dois rankings.

A partir da terceira posição, os resultados divergem: Maringá ocupa o terceiro lugar em número de vendas, enquanto Blumenau assume essa posição em receita. Isso demonstra que o volume de transações não determina sozinho o desempenho financeiro de cada cidade.


## 9. Relatório Consolidado

Os principais resultados da análise são reunidos em um relatório JSON. A estrutura consolida indicadores gerais, desempenho mensal, destaques comerciais, segmentação de clientes e estatísticas das receitas.

### 9.1 Indicadores Gerais

O resumo apresenta a dimensão da análise, o volume vendido, a receita total e o ticket médio por venda.

In [27]:
numero_vendas = int(df["id_venda"].nunique())
receita_total = float(df["receita_total"].sum())

indicadores_gerais = {
    "registros_analisados": int(len(df)),
    "vendas_unicas": numero_vendas,
    "clientes_unicos": int(df["id_cliente"].nunique()),
    "unidades_vendidas": int(df["quantidade"].sum()),
    "receita_total": round(receita_total, 2),
    "ticket_medio": round(receita_total / numero_vendas, 2),
}

exibir_tabela(
    pd.DataFrame([indicadores_gerais]),
    formatos={
        "receita_total": "R$ {:,.2f}",
        "ticket_medio": "R$ {:,.2f}",
    },
)

registros_analisados,vendas_unicas,clientes_unicos,unidades_vendidas,receita_total,ticket_medio
5361,5361,534,12927,"R$ 10,338,017.17","R$ 1,928.37"


### 9.2 Desempenho Mensal

Os resultados mensais são convertidos em uma lista de dicionários compatível com o formato JSON.

In [28]:
desempenho_mensal = [
    {
        "mes": str(linha.mes_venda),
        "receita_total": round(float(linha.receita_total), 2),
        "unidades_vendidas": int(linha.unidades_vendidas),
        "numero_vendas": int(linha.numero_vendas),
    }
    for linha in metricas_gerais["por_mes"].itertuples(index=False)
]

### 9.3 Destaques Comerciais

Os líderes de receita por mês, produto, categoria e região são reunidos com o cliente de maior gasto.

In [29]:
mes_destaque = metricas_gerais["por_mes"].loc[
    metricas_gerais["por_mes"]["receita_total"].idxmax()
]

produto_destaque = metricas_gerais["por_produto"].iloc[0]
categoria_destaque = metricas_gerais["por_categoria"].iloc[0]
regiao_destaque = metricas_gerais["por_regiao"].iloc[0]
cliente_destaque = clientes_segmentados.iloc[0]

destaques_comerciais = {
    "mes_maior_receita": {
        "mes": str(mes_destaque["mes_venda"]),
        "receita_total": round(float(mes_destaque["receita_total"]), 2),
    },
    "produto_maior_receita": {
        "produto": str(produto_destaque["produto"]),
        "receita_total": round(float(produto_destaque["receita_total"]), 2),
    },
    "categoria_maior_receita": {
        "categoria": str(categoria_destaque["categoria"]),
        "receita_total": round(float(categoria_destaque["receita_total"]), 2),
    },
    "regiao_maior_receita": {
        "regiao": str(regiao_destaque["regiao"]),
        "receita_total": round(float(regiao_destaque["receita_total"]), 2),
        "numero_vendas": int(regiao_destaque["numero_vendas"]),
        "ticket_medio": round(float(regiao_destaque["ticket_medio"]), 2),
    },
    "cliente_maior_gasto": {
        "id_cliente": str(cliente_destaque["id_cliente"]),
        "nome_cliente": str(cliente_destaque["nome_cliente"]),
        "gasto_total": round(float(cliente_destaque["gasto_total"]), 2),
        "segmento": str(cliente_destaque["segmento"]),
    },
}

### 9.4 Segmentação e Estatísticas

A distribuição dos clientes e as estatísticas calculadas com NumPy são preparadas para compor o relatório.

In [30]:
segmentacao_clientes = [
    {
        "segmento": str(linha.segmento),
        "numero_clientes": int(linha.numero_clientes),
        "percentual": round(float(linha.percentual), 2),
    }
    for linha in distribuicao_segmentos.itertuples(index=False)
]

resumo_estatistico = {
    chave: round(float(valor), 2) for chave, valor in estatisticas_receita.items()
}

resumo_estatistico["vendas_acima_media"] = {
    "quantidade": int(quantidade_acima_media),
    "percentual": round(float(percentual_acima_media), 2),
}

### 9.5 Exportação do Relatório

As estruturas anteriores são consolidadas e gravadas em `reports/relatorio_vendas.json`.

In [31]:
relatorio_vendas = {
    "metadados": {
        "projeto": "SalesInsight PY",
        "arquivo_origem": caminho_dados.name,
        "periodo_analisado": {
            "inicio": df["data_venda"].min().strftime("%Y-%m-%d"),
            "fim": df["data_venda"].max().strftime("%Y-%m-%d"),
        },
    },
    "indicadores_gerais": indicadores_gerais,
    "desempenho_mensal": desempenho_mensal,
    "destaques_comerciais": destaques_comerciais,
    "segmentacao_clientes": segmentacao_clientes,
    "estatisticas_receitas": resumo_estatistico,
}

pasta_relatorios = Path("../reports")
pasta_relatorios.mkdir(
    parents=True,
    exist_ok=True,
)

caminho_relatorio = pasta_relatorios / "relatorio_vendas.json"

with caminho_relatorio.open(
    mode="w",
    encoding="utf-8",
) as arquivo:
    json.dump(
        relatorio_vendas,
        arquivo,
        ensure_ascii=False,
        indent=2,
    )

print(f"Relatório JSON salvo em: {caminho_relatorio}")

Relatório JSON salvo em: ../reports/relatorio_vendas.json


### 9.6 Leitura e Validação

O arquivo exportado é relido para confirmar sua integridade e verificar a consistência entre o total de clientes segmentados, a receita calculada e os indicadores gerais.

In [32]:
with caminho_relatorio.open(
    mode="r",
    encoding="utf-8",
) as arquivo:
    relatorio_validado = json.load(arquivo)

total_clientes_segmentados = sum(
    segmento["numero_clientes"]
    for segmento in relatorio_validado["segmentacao_clientes"]
)

if total_clientes_segmentados != indicadores_gerais["clientes_unicos"]:
    raise ValueError("A segmentação não corresponde ao total de clientes.")

if (
    relatorio_validado["estatisticas_receitas"]["soma"]
    != indicadores_gerais["receita_total"]
):
    raise ValueError("A receita total diverge das estatísticas.")

print("Relatório JSON lido e validado com sucesso.")
print(f"Seções: {list(relatorio_validado)}")
print(f"Clientes segmentados: {total_clientes_segmentados}")
print(
    json.dumps(
        relatorio_validado,
        ensure_ascii=False,
        indent=2,
    )
)

Relatório JSON lido e validado com sucesso.
Seções: ['metadados', 'indicadores_gerais', 'desempenho_mensal', 'destaques_comerciais', 'segmentacao_clientes', 'estatisticas_receitas']
Clientes segmentados: 534
{
  "metadados": {
    "projeto": "SalesInsight PY",
    "arquivo_origem": "vendas.csv",
    "periodo_analisado": {
      "inicio": "2025-01-01",
      "fim": "2025-06-30"
    }
  },
  "indicadores_gerais": {
    "registros_analisados": 5361,
    "vendas_unicas": 5361,
    "clientes_unicos": 534,
    "unidades_vendidas": 12927,
    "receita_total": 10338017.17,
    "ticket_medio": 1928.37
  },
  "desempenho_mensal": [
    {
      "mes": "Janeiro",
      "receita_total": 1299266.31,
      "unidades_vendidas": 1968,
      "numero_vendas": 824
    },
    {
      "mes": "Fevereiro",
      "receita_total": 1591691.83,
      "unidades_vendidas": 2063,
      "numero_vendas": 855
    },
    {
      "mes": "Março",
      "receita_total": 2291715.46,
      "unidades_vendidas": 2443,
      "n